# Three RCP pathways, many possible futures

**Time:** 35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Distinguish a forcing pathway from a model; interpret baselines and intervals; compare scenarios without treating them as probabilities.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/02_three_rcp_pathways.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [1]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

Ready: public poster aggregates + clearly labeled fictional facility fixtures.


## What the numbers mean

RCP 2.6, RCP 4.5 and RCP 8.5 label representative radiative-forcing pathways, conventionally expressed in W/m² near 2100. They are not degrees of warming, metres of sea-level rise or probabilities. Multiple physical models can be run under the same pathway. RCP 8.5 is a high-forcing scenario, not an assertion that it is the most likely future.

Modern SSP-based assessments add a socioeconomic framework. SSP1-2.6 is not an interchangeable replacement for RCP 2.6: model generation, baselines and assumptions also matter. The book keeps the poster's historical RCP framework and links to [IPCC AR6](https://www.ipcc.ch/report/ar6/wg1/chapter/summary-for-policymakers/) for context.

## Read a historical assessment carefully

The reference CSV below transcribes six rows of **IPCC AR5 WGI Table SPM.2**: global mean sea-level change relative to **1986–2005**, averaged over **2046–2065** or **2081–2100**. These are period averages, not exact-year local predictions. They are educational context, not inputs used to recreate the poster's PAT calculations.

In [2]:
ref = pd.read_csv(DATA / 'ipcc_ar5_reference.csv')
display(ref)
late = ref.query("period == '2081-2100'")
fig = go.Figure()
for _,r in late.iterrows():
    fig.add_trace(go.Scatter(x=[r.scenario],y=[r.mean_m],mode='markers',name=r.scenario,
        marker=dict(size=14,color=COLORS[r.scenario]),
        error_y=dict(type='data',symmetric=False,array=[r.likely_high_m-r.mean_m],arrayminus=[r.mean_m-r.likely_low_m])))
fig.update_layout(title='IPCC AR5: global change, 2081–2100 average',yaxis_title='Metres relative to 1986–2005',showlegend=False)
show(fig)

,scenario,period,baseline,mean_m,likely_low_m,likely_high_m,source
0,RCP 2.6,2046-2065,1986-2005,0.24,0.17,0.32,IPCC AR5 WGI Table SPM.2
1,RCP 4.5,2046-2065,1986-2005,0.26,0.19,0.33,IPCC AR5 WGI Table SPM.2
2,RCP 8.5,2046-2065,1986-2005,0.30,0.22,0.38,IPCC AR5 WGI Table SPM.2
3,RCP 2.6,2081-2100,1986-2005,0.40,0.26,0.55,IPCC AR5 WGI Table SPM.2
4,RCP 4.5,2081-2100,1986-2005,0.47,0.32,0.63,IPCC AR5 WGI Table SPM.2
5,RCP 8.5,2081-2100,1986-2005,0.63,0.45,0.82,IPCC AR5 WGI Table SPM.2


<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="9b9daf41-0c84-49a1-9518-788fba83e188" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("9b9daf41-0c84-49a1-9518-788fba83e188")) { Plotly.newPlot( "9b9daf41-0c84-49a1-9518-788fba83e188", [{"error_y":{"array":[0.15000000000000002],"arrayminus":[0.14],"symmetric":false,"type":"data"},"marker":{"color":"#087e8b","size":14},"mode":"markers","name":"RCP 2.6","x":["RCP 2.6"],"y":[0.4],"type":"scatter"},{"error_y":{"array":[0.16000000000000003],"arrayminus":[0.14999999999999997],"symmetric":false,"type":"data"},"marker":{"color":"#b26b00","size":14},"mode":"markers","name":"RCP 4.5","x":["RCP 4.5"],"y":[0.47],"type":"scatter"},{"error_y":{"array":[0.18999999999999995],"arrayminus":[0.18],"symmetric":false,"type":"data"},"marker":{"color":"#cb4260","size":14},"mode":"markers","name":"RCP 8.5","x":["RCP 8.5"],"y":[0.63],"type":"scatter"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"surface","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"mesh3d":[{"type":"mesh3d","colorbar":{"outlinewidth":0,"ticks":""}}],"scatter":[{"fillpattern":{"fillmode":"overlay","size":10,"solidity":0.2},"type":"scatter"}],"parcoords":[{"type":"parcoords","line":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolargl":[{"type":"scatterpolargl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"bar":[{"error_x":{"color":"#2a3f5f"},"error_y":{"color":"#2a3f5f"},"marker":{"line":{"color":"#E5ECF6","width":0.5},"pattern":{"fillmode":"overlay","size":10,"solidity":0.2}},"type":"bar"}],"scattergeo":[{"type":"scattergeo","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"s

The ranges overlap. A pathway difference is not the only uncertainty: ice dynamics, ocean change, land motion and elevation errors can matter locally. A Bangkok facility cannot be classified by comparing its height to a global average unless the vertical datums, local processes and flood processes are made consistent.

## A deliberately simple teaching curve

For interactive labs we invent a 2020 baseline and endpoints of 0.45, 0.65 and 1.00 m in 2100. With $t=(y-2020)/80$, the curve is $0.2t+(H-0.2)t^2$. These chosen values are **not** the AR5 table, a fitted forecast, Kopp outputs or Climate Central data.

In [3]:
toy = pd.DataFrame([{'scenario':s,'year':y,'toy_slr_m':toy_slr(s,y)} for s in SCENARIOS for y in YEARS])
fig=px.line(toy,x='year',y='toy_slr_m',color='scenario',markers=True,color_discrete_map=COLORS,
            title='Invented teaching curves — fictional 2020 baseline')
show(fig)

<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="acb2c18f-8616-4da8-baf8-d47cc8861ba9" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("acb2c18f-8616-4da8-baf8-d47cc8861ba9")) { Plotly.newPlot( "acb2c18f-8616-4da8-baf8-d47cc8861ba9", [{"hovertemplate":"scenario=RCP 2.6\u003cbr\u003eyear=%{x}\u003cbr\u003etoy_slr_m=%{y}\u003cextra\u003e\u003c\u002fextra\u003e","legendgroup":"RCP 2.6","line":{"color":"#087e8b","dash":"solid"},"marker":{"symbol":"circle"},"mode":"lines+markers","name":"RCP 2.6","orientation":"v","showlegend":true,"x":{"dtype":"i2","bdata":"5AfuB\u002fgHAggMCBYIIAgqCDQI"},"xaxis":"x","y":{"dtype":"f8","bdata":"AAAAAAAAAACamZmZmZmdP83MzMzMzLA\u002fNDMzMzMzvD\u002fNzMzMzMzEPwAAAAAAgMw\u002fmpmZmZmZ0j80MzMzM3PXP83MzMzMzNw\u002f"},"yaxis":"y","type":"scatter"},{"hovertemplate":"scenario=RCP 4.5\u003cbr\u003eyear=%{x}\u003cbr\u003etoy_slr_m=%{y}\u003cextra\u003e\u003c\u002fextra\u003e","legendgroup":"RCP 4.5","line":{"color":"#b26b00","dash":"solid"},"marker":{"symbol":"circle"},"mode":"lines+markers","name":"RCP 4.5","orientation":"v","showlegend":true,"x":{"dtype":"i2","bdata":"5AfuB\u002fgHAggMCBYIIAgqCDQI"},"xaxis":"x","y":{"dtype":"f8","bdata":"AAAAAAAAAABnZmZmZmagPwAAAAAAALQ\u002fNDMzMzOzwT80MzMzMzPLPwAAAAAAQNM\u002fzszMzMzM2T8AAAAAAKDgP83MzMzMzOQ\u002f"},"yaxis":"y","type":"scatter"},{"hovertemplate":"scenario=RCP 8.5\u003cbr\u003eyear=%{x}\u003cbr\u003etoy_slr_m=%{y}\u003cextra\u003e\u003c\u002fextra\u003e","legendgroup":"RCP 8.5","line":{"color":"#cb4260","dash":"solid"},"marker":{"symbol":"circle"},"mode":"lines+markers","name":"RCP 8.5","orientation":"v","showlegend":true,"x":{"dtype":"i2","bdata":"5AfuB\u002fgHAggMCBYIIAgqCDQI"},"xaxis":"x","y":{"dtype":"f8","bdata":"AAAAAAAAAAA0MzMzMzOjP5qZmZmZmbk\u002fAQAAAAAAyD80MzMzMzPTPwAAAAAAANw\u002fNDMzMzMz4z80MzMzMzPpPwAAAAAAAPA\u002f"},"yaxis":"y","type":"scatter"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"su

## Lab: compare like with like

Write a function that rejects combining projection tables when their baseline or period differs. Then explain why the near-term ordering in the published poster is 108, 107 and 108 in 2030. Do not force the historical data to be monotonic across scenarios just because the teaching curves are.

**Worked interpretation:** Scenario differences can be small near-term and do not override source-specific uncertainties and processing. The published 107 should remain 107. Explain the difference rather than replacing it with 108.

**Deliverable:** A labeled interval plot and a caption naming the source, vertical unit, baseline and time window.

Reference: [IPCC AR5 WGI Summary for Policymakers, Table SPM.2](https://www.ipcc.ch/site/assets/uploads/2018/02/WG1AR5_SPM_FINAL.pdf).